# MIE 446 — Aerospace Structures Wing Designer

**Fall 2026 | Team project | Guided design tool | Python + CadQuery + 3D printing**

You do **not** need to write CAD code in this notebook. Complete the single student input form, then run the notebook. The tool interprets the NACA airfoil, calculates the wing quantities, draws the airfoil and planform, creates the fit coupon, builds and validates the 3D wing, and downloads the print package.

Before starting, use **File → Save a copy in Drive** so your completed form persists. Use **Coupon Only** first. After physically testing the course-issued rod, change the form to **Final Wing**, record the coupon result, increment the revision, and run again. This is a **non-flying fabrication demonstrator**; no flight or load-capacity claim is permitted.

## 1. Start the design tool

This cell runs automatically at the start of each **Run all**. It installs the fixed course release and checks that the CAD environment is working.

In [ ]:
#@title Run this setup cell { display-mode: "form" }
from datetime import date
from pathlib import Path
import subprocess
import sys
import tempfile

COURSE_RELEASE = "v1.1.0"
REPO_URL = "https://github.com/Ehsan-Roohi/Aerospace-Structures.git"
REPO_DIR = Path("/content/Aerospace-Structures")

if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", COURSE_RELEASE, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", "tag", COURSE_RELEASE],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "checkout", "--force", COURSE_RELEASE],
        check=True,
    )
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)],
    check=True,
)
sys.path.insert(0, str(REPO_DIR / "src"))

from importlib.metadata import version
from IPython.display import Markdown, display
import cadquery as cq
from mie446_wing import (
    WingDesign,
    WingProject,
    WingProjectError,
    make_ai_log,
    plot_design_overview,
    plot_modules,
)

smoke = cq.Workplane("XY").box(20, 30, 4).faces(">Z").workplane().hole(6)
assert smoke.val().isValid()

def download_in_colab(path):
    try:
        from google.colab import files
        files.download(str(path))
    except ImportError:
        print("File remains at:", path)

display(Markdown(
    f"✅ **Design tool ready.** Course release `{COURSE_RELEASE}`, "
    f"CadQuery `{version('cadquery')}`, OCP `{version('cadquery-ocp')}`."
))

## 2. Enter your wing design

This is the **only form you edit**. All length entries are in millimetres; NACA code and module count are dimensionless. `x/c` means chordwise distance divided by the local chord. Use positive dimensions, tip chord ≤ root chord ≤ 300 mm, semi-span ÷ modules ≤ 300 mm, and skin/rib thickness ≥ 0.8 mm. The physical model is one semi-wing; full-wing quantities are analytical equivalents only. Course-fixed settings are two 4 mm rods at `x/c = 0.30` and `0.60`, 1.2 mm sleeve wall, 1.2 mm trailing edge, three interior ribs, regular PLA Pro, and a 300 g CAD-mass limit.

In [ ]:
#@title STUDENT INPUT FORM { display-mode: "form" }
#@markdown ### A. Team and workflow
WORKFLOW_STAGE = "Coupon Only"  #@param ["Coupon Only", "Final Wing"]
TEAM_ID = "Team00"  #@param {type:"string"}
TEAM_MEMBERS = ""  #@param {type:"string"}
REVISION = "R01"  #@param {type:"string"}

#@markdown ### B. Airfoil and wing geometry
NACA_CODE = "2412"  #@param {type:"string"}
SEMI_SPAN_MM = 450.0  #@param {type:"number"}
ROOT_CHORD_MM = 160.0  #@param {type:"number"}
TIP_CHORD_MM = 100.0  #@param {type:"number"}
SKIN_THICKNESS_MM = 1.2  #@param {type:"number"}
RIB_THICKNESS_MM = 1.6  #@param {type:"number"}
MODULE_COUNT = 3  #@param {type:"slider", min:2, max:3, step:1}

#@markdown ### C. Physical fit-coupon record — complete for Final Wing
SELECTED_RADIAL_CLEARANCE_MM = 0.25  #@param [0.15, 0.25, 0.35] {type:"raw"}
COUPON_CONFIRMED = False  #@param {type:"boolean"}
COUPON_REVISION = "R01"  #@param {type:"string"}
MEASURED_ROD_DIAMETER_MM = 4.0  #@param {type:"number"}
COUPON_FIT_RESULT = "Not tested yet"  #@param ["Not tested yet", "Free/slip fit selected", "Snug fit selected"]
COUPON_TESTER = ""  #@param {type:"string"}
COUPON_DATE = "YYYY-MM-DD"  #@param {type:"string"}

#@markdown ### D. Engineering understanding
AIRFOIL_CHOICE_REASON = ""  #@param {type:"string"}
COMPARISON_TIP_CHORD_MM = 90.0  #@param {type:"number"}
PREDICTED_AREA_CHANGE = "Choose before running"  #@param ["Choose before running", "increase", "decrease", "no change"]
PREDICTED_ASPECT_RATIO_CHANGE = "Choose before running"  #@param ["Choose before running", "increase", "decrease", "no change"]
PREDICTION_EXPLANATION = ""  #@param {type:"string"}
RESULT_INTERPRETATION = ""  #@param {type:"string"}
PRINT_DEFECT_CODE_CANNOT_DETECT = ""  #@param {type:"string"}

#@markdown ### E. AI use and independent verification
#@markdown If there were several material uses, summarize them with semicolons.
AI_USED = False  #@param {type:"boolean"}
AI_TOOL = ""  #@param {type:"string"}
AI_PURPOSE = ""  #@param {type:"string"}
AI_AFFECTED_ITEM = ""  #@param {type:"string"}
STUDENT_CHANGE_TO_AI_OUTPUT = ""  #@param {type:"string"}
INDEPENDENT_CHECK = ""  #@param {type:"string"}
AI_ERROR_OR_LIMITATION = ""  #@param {type:"string"}
AI_VERDICT = "Accept with Limitations"  #@param ["Accept", "Accept with Limitations", "Reject"]

# Reset every downstream result whenever the form is rerun.
project = None
analysis = None
comparison = None
coupon_ready = False
coupon_record = None
built_project = None
try:
    design = WingDesign(
        naca=NACA_CODE,
        semi_span_mm=SEMI_SPAN_MM,
        root_chord_mm=ROOT_CHORD_MM,
        tip_chord_mm=TIP_CHORD_MM,
        skin_mm=SKIN_THICKNESS_MM,
        rib_thickness_mm=RIB_THICKNESS_MM,
        module_count=int(MODULE_COUNT),
        rod_clearance_mm=float(SELECTED_RADIAL_CLEARANCE_MM),
    )
    project = WingProject(design, team_id=TEAM_ID, revision=REVISION)
    display(Markdown("✅ **Form accepted.** Continue to Step 3."))
except WingProjectError as error:
    display(Markdown(f"❌ **Correct the input form:** {error}"))

## 3. Preview and understand the design

The notebook now interprets the NACA code, calculates area, taper ratio, aspect ratio and mean aerodynamic chord, checks the inputs, and draws the root airfoil and wing planform. No 3D CAD is built yet.

In [ ]:
#@title Run the automatic 2D analysis { display-mode: "form" }
analysis = None
comparison = None
if project is None:
    display(Markdown("⚠️ Step 3 skipped. Correct and rerun the Step 2 form."))
else:
    try:
        analysis = project.analyze()
        display(Markdown(analysis.to_markdown()))
        plot_design_overview(analysis.parameters).show()
        if (
            PREDICTED_AREA_CHANGE == "Choose before running"
            or PREDICTED_ASPECT_RATIO_CHANGE == "Choose before running"
            or not PREDICTION_EXPLANATION.strip()
        ):
            display(Markdown(
                "⚠️ **Prediction not yet recorded.** Complete the three prediction fields in Step 2, "
                "rerun Step 2, and then rerun Step 3 to reveal the comparison."
            ))
        else:
            comparison = analysis.compare_tip_chord(COMPARISON_TIP_CHORD_MM)
            display(Markdown(comparison.to_markdown()))
            area_mark = "✅" if PREDICTED_AREA_CHANGE == comparison.area_direction else "❌"
            ar_mark = "✅" if PREDICTED_ASPECT_RATIO_CHANGE == comparison.aspect_ratio_direction else "❌"
            display(Markdown(
                f"{area_mark} Area prediction: **{PREDICTED_AREA_CHANGE}**; "
                f"{ar_mark} aspect-ratio prediction: **{PREDICTED_ASPECT_RATIO_CHANGE}**."
            ))
    except WingProjectError as error:
        display(Markdown(f"❌ **Design analysis stopped:** {error}"))

## 4. Print and record the rod-fit coupon

In **Coupon Only** mode, this step creates and downloads the three-hole clearance coupon, its wing-parameter snapshot and its mapping. One, two and three top-face dimples identify the 0.15, 0.25 and 0.35 mm radial-clearance holes. Print it using the assigned K2 Pro / 0.4 mm / regular PLA Pro profile. Test the measured course rod without forcing it. For the next run, select **Final Wing** and complete every coupon-record field in Step 2.

In [ ]:
#@title Run the coupon stage { display-mode: "form" }
coupon_ready = False
coupon_record = None
if project is None:
    display(Markdown("⚠️ Step 4 skipped because the design form is not valid."))
elif WORKFLOW_STAGE == "Coupon Only":
    try:
        coupon_root = Path(tempfile.mkdtemp(prefix=f"mie446_{TEAM_ID}_{REVISION}_coupon_"))
        coupon_package = project.make_coupon(coupon_root)
        display(Markdown(
            f"✅ **Coupon package ready:** `{coupon_package.archive_path.name}`. "
            "Download it, print it, and stop after this run."
        ))
        download_in_colab(coupon_package.archive_path)
    except WingProjectError as error:
        display(Markdown(f"❌ **Coupon export stopped:** {error}"))
else:
    missing_coupon_fields = []
    if not COUPON_CONFIRMED:
        missing_coupon_fields.append("COUPON_CONFIRMED")
    if COUPON_FIT_RESULT == "Not tested yet":
        missing_coupon_fields.append("COUPON_FIT_RESULT")
    if not COUPON_TESTER.strip():
        missing_coupon_fields.append("COUPON_TESTER")
    try:
        date.fromisoformat(COUPON_DATE.strip())
    except ValueError:
        missing_coupon_fields.append("COUPON_DATE")
    if not COUPON_REVISION.strip():
        missing_coupon_fields.append("COUPON_REVISION")
    elif COUPON_REVISION.strip().casefold() == REVISION.strip().casefold():
        missing_coupon_fields.append("REVISION (must differ from COUPON_REVISION)")
    if not 3.5 <= float(MEASURED_ROD_DIAMETER_MM) <= 4.5:
        missing_coupon_fields.append("MEASURED_ROD_DIAMETER_MM (expected 3.5–4.5 mm)")
    if missing_coupon_fields:
        display(Markdown(
            "⚠️ **Final Wing is waiting for the physical coupon record.** Complete: `"
            + "`, `".join(missing_coupon_fields)
            + "`, then rerun Steps 2–4."
        ))
    else:
        coupon_record = {
            "confirmed": True,
            "team_id": TEAM_ID,
            "coupon_revision": COUPON_REVISION.strip(),
            "measured_rod_diameter_mm": float(MEASURED_ROD_DIAMETER_MM),
            "selected_radial_clearance_mm": float(SELECTED_RADIAL_CLEARANCE_MM),
            "fit_result": COUPON_FIT_RESULT,
            "tester": COUPON_TESTER.strip(),
            "date": COUPON_DATE.strip(),
        }
        coupon_ready = True
        display(Markdown(
            f"✅ **Coupon record accepted.** Selected radial clearance: "
            f"**{SELECTED_RADIAL_CLEARANCE_MM:.2f} mm**."
        ))

## 5. Build and verify the 3D wing

This step runs only in **Final Wing** mode after the coupon record is complete. CadQuery builds the complete semi-wing, adds ribs and rod sleeves, cuts the holes, divides the wing into printable modules, and performs all geometry checks automatically.

In [ ]:
#@title Build and inspect the 3D model { display-mode: "form" }
built_project = None
if WORKFLOW_STAGE == "Coupon Only":
    display(Markdown("ℹ️ **3D wing intentionally skipped.** Finish the physical coupon test first."))
elif project is None:
    display(Markdown("⚠️ Step 5 skipped because the design form is not valid."))
elif TEAM_ID == "Team00" or not TEAM_MEMBERS.strip():
    display(Markdown("⚠️ Step 5 skipped. Enter the real team ID and team members in Step 2."))
elif not coupon_ready:
    display(Markdown("⚠️ Step 5 skipped until Step 4 accepts the coupon record."))
else:
    try:
        display(Markdown("⏳ Building the wing; this usually takes about one minute..."))
        built_project = project.build(coupon_confirmed=True)
        display(Markdown(built_project.to_markdown()))
        plot_modules(
            built_project.build.modules,
            title=f"MIE 446 {TEAM_ID} {REVISION} — printable modules",
        ).show()
    except WingProjectError as error:
        display(Markdown(f"❌ **3D build stopped:** {error}"))

## 6. Export and download the submission

In Final Wing mode, the notebook checks the required engineering responses and AI disclosure, then creates a verified ZIP containing STEP, STL, 3MF, parameters, calculations, validation, the self-reported coupon record, AI log, design summary, an interactive design plot and SHA-256 hashes.

In [ ]:
#@title Create and download the final ZIP { display-mode: "form" }
if WORKFLOW_STAGE == "Coupon Only":
    display(Markdown(
        "✅ **Coupon run complete.** Print and test the coupon. Then choose Final Wing in Step 2, "
        "increment the revision, complete the coupon record, and run all again."
    ))
elif built_project is None:
    display(Markdown("⚠️ Final export skipped because no validated 3D wing is available."))
elif built_project.project != project:
    display(Markdown("⚠️ The form changed after the 3D build. Rerun Steps 2–6 before export."))
else:
    missing_submission_fields = []
    required_text = {
        "TEAM_MEMBERS": TEAM_MEMBERS,
        "AIRFOIL_CHOICE_REASON": AIRFOIL_CHOICE_REASON,
        "PREDICTION_EXPLANATION": PREDICTION_EXPLANATION,
        "RESULT_INTERPRETATION": RESULT_INTERPRETATION,
        "PRINT_DEFECT_CODE_CANNOT_DETECT": PRINT_DEFECT_CODE_CANNOT_DETECT,
    }
    missing_submission_fields.extend(
        name for name, value in required_text.items() if not value.strip()
    )
    if TEAM_ID == "Team00":
        missing_submission_fields.append("TEAM_ID")
    if PREDICTED_AREA_CHANGE == "Choose before running":
        missing_submission_fields.append("PREDICTED_AREA_CHANGE")
    if PREDICTED_ASPECT_RATIO_CHANGE == "Choose before running":
        missing_submission_fields.append("PREDICTED_ASPECT_RATIO_CHANGE")
    if missing_submission_fields:
        display(Markdown(
            "⚠️ **Submission is not complete.** Fill: `"
            + "`, `".join(missing_submission_fields)
            + "`, then rerun Steps 2–6."
        ))
    else:
        try:
            ai_log = make_ai_log(
                ai_used=AI_USED,
                tool=AI_TOOL,
                purpose=AI_PURPOSE,
                affected_code_or_claim=AI_AFFECTED_ITEM,
                student_change=STUDENT_CHANGE_TO_AI_OUTPUT,
                independent_check=INDEPENDENT_CHECK,
                error_or_limitation_found=AI_ERROR_OR_LIMITATION,
                verdict=AI_VERDICT,
            )
            automatic_comparison = built_project.analysis.compare_tip_chord(
                COMPARISON_TIP_CHORD_MM
            )
            student_record = {
                "course_release": COURSE_RELEASE,
                "team_id": TEAM_ID,
                "team_members": TEAM_MEMBERS,
                "revision": REVISION,
                "workflow_stage": WORKFLOW_STAGE,
                "coupon_record": coupon_record,
                "engineering_responses": {
                    "airfoil_choice_reason": AIRFOIL_CHOICE_REASON,
                    "comparison_tip_chord_mm": float(COMPARISON_TIP_CHORD_MM),
                    "predicted_area_change": PREDICTED_AREA_CHANGE,
                    "predicted_aspect_ratio_change": PREDICTED_ASPECT_RATIO_CHANGE,
                    "computed_area_change": automatic_comparison.area_direction,
                    "computed_area_change_mm2": automatic_comparison.full_area_change_mm2,
                    "computed_aspect_ratio_change": automatic_comparison.aspect_ratio_direction,
                    "computed_aspect_ratio_delta": automatic_comparison.aspect_ratio_change,
                    "area_prediction_matches": PREDICTED_AREA_CHANGE == automatic_comparison.area_direction,
                    "aspect_ratio_prediction_matches": PREDICTED_ASPECT_RATIO_CHANGE == automatic_comparison.aspect_ratio_direction,
                    "prediction_explanation": PREDICTION_EXPLANATION,
                    "result_interpretation": RESULT_INTERPRETATION,
                    "print_defect_code_cannot_detect": PRINT_DEFECT_CODE_CANNOT_DETECT,
                },
                "ai_use_declaration": (
                    "Material AI use disclosed in ai_use_log.json"
                    if AI_USED
                    else "Team declares no material AI use"
                ),
            }
            export_root = Path(tempfile.mkdtemp(prefix=f"mie446_{TEAM_ID}_{REVISION}_final_"))
            submission = built_project.export_submission(
                export_root,
                ai_log=ai_log,
                student_record=student_record,
            )
            display(Markdown(
                f"✅ **Submission package ready:** `{submission.archive_path.name}` with "
                f"**{len(submission.manifest['files'])} hashed artifacts**."
            ))
            download_in_colab(submission.archive_path)
        except WingProjectError as error:
            display(Markdown(f"❌ **Final export stopped:** {error}"))

**After download:** Import every module into the current staff-approved Creality K2 Pro / 0.4 mm / regular PLA Pro profile. Confirm millimetres and 100% scale, inspect every layer, supervise the first layers, measure the printed parts, and submit the verified ZIP. Passing the notebook does not certify printability, structural performance or flight safety.